# Notebook 10: ReAct Agents vs RLM

**ReAct** (Reasoning + Acting) is the dominant agent paradigm. The model alternates between:
- **Thought**: Reasoning about what to do next
- **Action**: Calling a tool or performing an operation
- **Observation**: Seeing the result

In this notebook you'll learn:
- How to build a simple ReAct agent from scratch
- The CodeAct variant (actions are code, not predefined tools)
- How ReAct compares to RLM on the same tasks
- The key difference: sequential vs programmatic

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
import json, os, re

from rlm_core import RLMEngine, Sandbox
from rlm_core.visualizer import tree_to_text

# Simulated client for demos
class SimulatedClient:
    def __init__(self):
        self.call_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
    
    def completion(self, prompt, **kwargs):
        self.call_count += 1
        self.total_prompt_tokens += len(prompt.split())
        self.total_completion_tokens += 15
        
        class Result:
            def __init__(self, text, pt, ct):
                self.text, self.prompt_tokens, self.completion_tokens = text, pt, ct
        
        # --- ReAct-style responses (Thought/Action/Final Answer) ---
        lower = prompt.lower()
        
        # Check if this is a ReAct-style prompt (has "Available tools" or "Respond with EXACTLY")
        if "available tools" in lower and "respond with exactly" in lower:
            if "secret" in lower and "code" in lower:
                if "observation" not in lower:
                    text = "Thought: I need to search for the secret code in the context.\nAction: search: secret code"
                else:
                    text = "Thought: I found the secret code from the search result.\nFinal Answer: BLUE-FALCON-42"
            elif "cto" in lower or "who" in lower:
                if "observation" not in lower:
                    text = "Thought: I need to search for the company that designed the SolarFlow battery.\nAction: search: SolarFlow"
                elif lower.count("observation") < 2:
                    text = "Thought: I found SolarFlow is by NovaTech. Now I need to find the CTO.\nAction: search: CTO"
                else:
                    text = "Thought: I found the CTO from the search results.\nFinal Answer: Sarah Martinez"
            else:
                text = "Thought: Let me search for the answer.\nAction: search: " + prompt.split("Question:")[-1].split("\n")[0].strip().split()[0]
        # --- CodeAct-style responses (Python code) ---
        elif "write python code" in lower or "print('final answer" in lower or 'print("final answer' in lower:
            if "secret" in lower or "code" in lower:
                text = """import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
print(f"FINAL ANSWER: {match.group(1) if match else 'Not found'}")"""
            else:
                text = 'print("FINAL ANSWER: Information processed")'
        # --- RLM-style responses (code with FINAL()) ---
        elif "secret" in lower or "code" in lower:
            text = '''import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
FINAL(match.group(1) if match else "Not found")'''
        elif "how many" in lower or "count" in lower or "red" in lower:
            text = '''import json
data = json.loads(context) if context.strip().startswith('[') else []
count = sum(1 for i in data if i.get("color")=="red" and i.get("size")=="large")
FINAL(f"{count}")'''
        elif "engineer" in lower or "who" in lower or "cto" in lower:
            text = '''import re
m = re.search(r'(Dr\\.? [A-Z][a-z]+ [A-Z][a-z]+|[A-Z][a-z]+ [A-Z][a-z]+).*(?:lead engineer|CTO)', context)
FINAL(m.group(1) if m else "Not found")'''
        else:
            text = 'FINAL("Information processed")'
        
        return Result(text, len(prompt.split()), 15)

client = SimulatedClient()
engine = RLMEngine(client=client, max_depth=3)

# Load all datasets
with open("../data/samples/needle_haystack.txt") as f:
    needle_doc = f.read()
with open("../data/samples/aggregation_items.json") as f:
    agg_items = json.load(f)
docs = []
doc_dir = "../data/samples/multihop_docs"
for fname in sorted(os.listdir(doc_dir)):
    if fname.endswith('.txt'):
        with open(os.path.join(doc_dir, fname)) as f:
            docs.append(f.read())

print(f"Loaded: needle ({len(needle_doc)} chars), {len(agg_items)} items, {len(docs)} docs")

## What is ReAct?

The ReAct loop:
```
Question -> Thought -> Action -> Observation -> Thought -> Action -> ... -> Final Answer
```

Each step is ONE thought + ONE action. The model can't plan multiple actions ahead -- it must wait for each observation before deciding the next step. This is **sequential**.

In [ ]:
class ReActAgent:
    """A simple ReAct agent with configurable tools."""
    
    def __init__(self, client, tools, max_steps=10):
        self.client = client
        self.tools = tools
        self.max_steps = max_steps
        self.trace = []  # Record of all thoughts, actions, observations
    
    def run(self, question, context=""):
        self.trace = []
        history = f"Question: {question}\n"
        if context:
            history += f"Context available ({len(context)} chars). Use the 'read' tool to examine it.\n"
        
        available = ", ".join(self.tools.keys())
        
        for step in range(self.max_steps):
            prompt = f"""{history}

Available tools: {available}

Respond with EXACTLY this format:
Thought: [your reasoning]
Action: [tool_name]: [input]

Or if you have the final answer:
Thought: [reasoning]
Final Answer: [answer]"""
            
            response = self.client.completion(prompt)
            text = response.text
            
            # Parse thought
            thought = ""
            if "Thought:" in text:
                thought = text.split("Thought:")[1].split("\n")[0].strip()
            
            # Check for final answer
            if "Final Answer:" in text:
                answer = text.split("Final Answer:")[1].strip()
                self.trace.append({"step": step+1, "thought": thought, "final_answer": answer})
                return answer
            
            # Parse action
            action_text = ""
            if "Action:" in text:
                action_text = text.split("Action:")[1].strip().split("\n")[0]
            
            # Execute action
            if ":" in action_text:
                tool_name, tool_input = action_text.split(":", 1)
                tool_name = tool_name.strip()
                tool_input = tool_input.strip()
                
                if tool_name in self.tools:
                    observation = self.tools[tool_name](tool_input, context)
                    self.trace.append({
                        "step": step+1, "thought": thought,
                        "action": f"{tool_name}: {tool_input}",
                        "observation": str(observation)[:200]
                    })
                    history += f"\nThought: {thought}\nAction: {tool_name}: {tool_input}\nObservation: {observation}\n"
                else:
                    history += f"\nObservation: Unknown tool '{tool_name}'\n"
            else:
                # No valid action, try to get final answer
                self.trace.append({"step": step+1, "thought": thought, "raw": text[:100]})
                return text.split("\n")[0] if text else "No answer found"
        
        return "Max steps reached"
    
    def print_trace(self):
        print("=== ReAct Trace ===")
        for entry in self.trace:
            print(f"Step {entry['step']}:")
            if 'thought' in entry and entry['thought']:
                print(f"  Thought: {entry['thought'][:80]}")
            if 'action' in entry:
                print(f"  Action: {entry['action'][:80]}")
            if 'observation' in entry:
                print(f"  Observation: {entry['observation'][:80]}")
            if 'final_answer' in entry:
                print(f"  Final Answer: {entry['final_answer']}")

# Define tools
def search_tool(query, context):
    import re
    matches = re.findall(f'[^.]*{re.escape(query.split()[0])}[^.]*\\.', context, re.IGNORECASE)
    return matches[0] if matches else "No match found"

def read_tool(section, context):
    try:
        start = int(section.split("-")[0]) if "-" in section else 0
        end = int(section.split("-")[1]) if "-" in section else 500
        return context[start:end]
    except:
        return context[:500]

def count_tool(query, context):
    return f"Found {context.lower().count(query.lower())} occurrences of '{query}'"

tools = {
    "search": search_tool,
    "read": read_tool,
    "count": count_tool,
}

agent = ReActAgent(client, tools, max_steps=5)
print("ReAct agent ready with tools: search, read, count")

## ReAct vs RLM: Needle-in-a-Haystack

In [ ]:
print("=== ReAct Agent ===")
react_answer = agent.run("What is the secret code?", needle_doc)
print(f"Answer: {react_answer}")
agent.print_trace()

print(f"\n=== RLM ===")
rlm_result = engine.run("What is the secret code?", needle_doc)
print(f"Answer: {rlm_result.answer}")
print(f"LLM calls: {rlm_result.total_llm_calls}")
print(tree_to_text(rlm_result.root_node))

## The Key Difference: Sequential vs Programmatic

```
ReAct:                          RLM:
Step 1: Think                   Step 1: Write complete program
Step 2: Act (search)              - regex search
Step 3: Observe result            - chunk if needed  
Step 4: Think again               - recursive sub-calls
Step 5: Act (read)                - aggregate results
Step 6: Observe                   - FINAL(answer)
Step 7: Think
Step 8: Final Answer

ONE action per step             MANY actions per code block
```

ReAct takes **N sequential LLM calls** for N actions. RLM writes **one program** that executes multiple operations.

## CodeAct: The Bridge Between ReAct and RLM

CodeAct agents write **code** as their actions instead of calling predefined tools. This is halfway to an RLM:

In [ ]:
class CodeActAgent:
    """A ReAct agent where actions are Python code."""
    
    def __init__(self, client, max_steps=5):
        self.client = client
        self.max_steps = max_steps
        self.sandbox = None
        self.trace = []
    
    def run(self, question, context):
        self.sandbox = Sandbox(variables={"context": context})
        self.trace = []
        history = f"Question: {question}\nContext is available as the variable 'context'.\n"
        
        for step in range(self.max_steps):
            prompt = f"""{history}
Write Python code to help answer the question. Use print() for output.
When you have the answer, print('FINAL ANSWER: <answer>')

```python
"""
            response = self.client.completion(prompt)
            code = response.text.replace("```", "").strip()
            
            result = self.sandbox.execute(code)
            
            self.trace.append({
                "step": step + 1,
                "code": code[:200],
                "output": result.stdout[:200] if result.stdout else "",
                "error": result.error[:100] if result.error else None,
            })
            
            if "FINAL ANSWER:" in (result.stdout or ""):
                answer = result.stdout.split("FINAL ANSWER:")[1].strip()
                return answer
            
            history += f"\nCode:\n{code}\nOutput: {result.stdout or result.error}\n"
        
        return "Max steps reached"
    
    def print_trace(self):
        for entry in self.trace:
            print(f"Step {entry['step']}:")
            print(f"  Code: {entry['code'][:100]}...")
            if entry['output']:
                print(f"  Output: {entry['output'][:100]}")
            if entry['error']:
                print(f"  Error: {entry['error'][:100]}")

codeact = CodeActAgent(client)
print("CodeAct agent ready!")
print("\nCodeAct is the bridge: ReAct + code = almost RLM")
print("But CodeAct is still SEQUENTIAL -- one code block per step.")
print("RLM writes ONE program with recursion, loops, and sub-calls.")

## Visualization: Linear Trace vs Recursion Tree

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ReAct trace (linear)
ax1 = axes[0]
ax1.set_title("ReAct: Sequential Steps", fontsize=14)
steps = ["Thought", "Action:\nsearch", "Observe", "Thought", "Action:\nread", "Observe", "Answer"]
for i, step in enumerate(steps):
    color = '#FF6B6B' if 'Thought' in step else '#4ECDC4' if 'Action' in step else '#45B7D1' if 'Observe' in step else '#FFD93D'
    ax1.add_patch(plt.Rectangle((0.3, 6-i), 0.4, 0.8, facecolor=color, edgecolor='black'))
    ax1.text(0.5, 6.4-i, step, ha='center', va='center', fontsize=9)
    if i < len(steps) - 1:
        ax1.annotate('', xy=(0.5, 6-i), xytext=(0.5, 6.2-i),
                     arrowprops=dict(arrowstyle='->', color='black'))
ax1.set_xlim(0, 1)
ax1.set_ylim(-0.5, 7.5)
ax1.axis('off')

# RLM tree
ax2 = axes[1]
ax2.set_title("RLM: Programmatic Tree", fontsize=14)
# Root
ax2.add_patch(plt.Rectangle((0.3, 5.5), 0.4, 0.8, facecolor='#4ECDC4', edgecolor='black'))
ax2.text(0.5, 5.9, "Root LLM\n(writes code)", ha='center', va='center', fontsize=9)
# Children
for i, (x, label) in enumerate([(0.1, "regex\nsearch"), (0.5, "chunk\nprocess"), (0.7, "sub-call")]):
    ax2.add_patch(plt.Rectangle((x, 3.5), 0.25, 0.8, facecolor='#45B7D1', edgecolor='black'))
    ax2.text(x+0.125, 3.9, label, ha='center', va='center', fontsize=8)
    ax2.annotate('', xy=(x+0.125, 4.3), xytext=(0.5, 5.5),
                 arrowprops=dict(arrowstyle='->', color='black'))
# Final
ax2.add_patch(plt.Rectangle((0.3, 1.5), 0.4, 0.8, facecolor='#FFD93D', edgecolor='black'))
ax2.text(0.5, 1.9, "FINAL(answer)", ha='center', va='center', fontsize=9)
ax2.annotate('', xy=(0.5, 2.3), xytext=(0.5, 3.5),
             arrowprops=dict(arrowstyle='->', color='black'))
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 7)
ax2.axis('off')

plt.tight_layout()
plt.show()

## Comparison Summary

| Feature | ReAct | CodeAct | RLM |
|---------|-------|---------|-----|
| Actions | Predefined tools | Code blocks | Code with recursion |
| Planning | One step at a time | One step at a time | Entire program at once |
| Parallelism | None (sequential) | None (sequential) | Yes (multiple sub-calls) |
| Flexibility | Limited by tools | High (any code) | Highest (code + self-calls) |
| Transparency | Very readable | Readable | Requires tree visualization |
| Latency | N sequential calls | N sequential calls | 1 program + M sub-calls |

## Exercise

Modify the ReAct agent to handle a multi-hop question requiring 3+ steps, then compare with RLM.

In [ ]:
# Exercise: Multi-hop with ReAct
multi_context = "\n\n---\n\n".join(docs)

print("=== ReAct: Multi-hop ===")
react_multi = agent.run(
    "Who is the CTO of the company that designed the SolarFlow battery?", 
    multi_context
)
print(f"Answer: {react_multi}")
agent.print_trace()

print(f"\n=== RLM: Multi-hop ===")
rlm_multi = engine.run(
    "Who is the CTO of the company that designed the SolarFlow battery?",
    multi_context
)
print(f"Answer: {rlm_multi.answer}")
print(f"Expected: Sarah Martinez")

## Latency Analysis: ReAct vs RLM

Let's quantify the sequential overhead of ReAct compared to the programmatic approach of RLM:

In [ ]:
# Latency comparison: sequential ReAct vs programmatic RLM
import matplotlib.pyplot as plt

# Simulate latency for different task complexities
task_steps = [1, 2, 3, 5, 8, 10]
llm_call_time_ms = 500  # Simulated per-call latency

react_latency = [n * llm_call_time_ms for n in task_steps]  # N sequential calls
rlm_latency = [1 * llm_call_time_ms + max(0, n-1) * 100 for n in task_steps]  # 1 program + fast sub-calls

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(task_steps, react_latency, 'o-', color='#FF6B6B', linewidth=2, markersize=8, label='ReAct (sequential)')
ax.plot(task_steps, rlm_latency, 's-', color='#4ECDC4', linewidth=2, markersize=8, label='RLM (programmatic)')

ax.set_xlabel('Number of Actions Required', fontsize=12)
ax.set_ylabel('Estimated Latency (ms)', fontsize=12)
ax.set_title('Latency Scaling: ReAct vs RLM', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate the gap
for i, n in enumerate(task_steps):
    if n >= 5:
        gap = react_latency[i] - rlm_latency[i]
        ax.annotate(f'{gap}ms faster',
                    xy=(n, rlm_latency[i]), xytext=(n + 0.3, rlm_latency[i] + 300),
                    arrowprops=dict(arrowstyle='->', color='gray'),
                    fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print("\nKey insight: ReAct latency grows LINEARLY with task complexity.")
print("RLM latency grows much slower -- one program handles multiple actions.")

## Key Takeaways

1. **ReAct** is sequential: Thought -> Action -> Observation, one step at a time
2. **CodeAct** bridges the gap: actions are code, not predefined tools
3. **RLM** is programmatic: writes a complete program with loops, branches, and recursion
4. **ReAct is more transparent** -- each step is human-readable
5. **RLM is more efficient** -- one program vs many sequential LLM calls
6. **RLM can parallelize** -- multiple sub-calls planned in one code block
7. Use ReAct when you need **auditability**; use RLM when you need **capability**

## What's Next?

Notebook 11 explores the connection to **DSPy** -- programming with LLMs, not prompting.